# 01 — Collecte des donnees produits (API Channel3)

**Objectif** : extraire un echantillon de produits mode et luxe sur 6 categories,
et le materialiser en `data/produits_bruts.csv`.

**Prerequis** : une cle d'API Channel3 dans un fichier `.env` a la racine du projet :

```
CHANNEL3_API_KEY=votre_cle
```

Les sorties de ce notebook sont volontairement vides : il appelle une API externe
authentifiee, donc il n'est pas rejouable en CI. La version scriptee et parametree
se trouve dans `src/collecte.py`.


In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("CHANNEL3_API_KEY")

if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

print("Clé chargée :", API_KEY is not None)
print("Longueur de la clé :", len(API_KEY) if API_KEY else 0)

In [ ]:
import os
from dotenv import load_dotenv
from channel3_sdk import Channel3

# Charger la clé
load_dotenv()
API_KEY = os.getenv("CHANNEL3_API_KEY")
if API_KEY:
    API_KEY = API_KEY.strip().strip('"').strip("'")

# Créer le client Channel3 (il s'authentifie avec ta clé)
client = Channel3(api_key=API_KEY)

# Notre première recherche : des sacs en cuir (produit luxe/mode typique)
page = client.products.search(query="leather handbag")

# Combien de produits reçus ?
print("Nombre de produits reçus :", len(page.products))

# Regardons le premier produit
premier = page.products[0]
print("\n--- Premier produit ---")
print("ID :", premier.id)
print("Titre :", premier.title)

In [ ]:
# Prenons le premier produit et regardons TOUT ce qu'il contient
premier = page.products[0]

print("=== STRUCTURE COMPLÈTE D'UN PRODUIT ===\n")
print("ID :", premier.id)
print("Titre :", premier.title)
print("Description :", premier.description)
print("Marques :", premier.brands)
print("Catégories :", premier.categories)
print("Matériaux :", premier.materials)
print("\n=== LES OFFRES (c'est là qu'est le PRIX) ===")
print(premier.offers)

In [ ]:
# Le prix est dans offers → price → price
premier = page.products[0]

if premier.offers:                        # s'il y a au moins une offre
    offre = premier.offers[0]             # la première offre
    print("Domaine (retailer) :", offre.domain)
    print("URL :", offre.url)
    print("Prix :", offre.price.price)
    print("Devise :", offre.price.currency)
    print("Prix barré (avant promo) :", offre.price.compare_at_price)
else:
    print("Pas d'offre pour ce produit")

In [ ]:
def extraire_infos(produit):
    """Prend un produit Channel3 et renvoie ses infos clés dans un dictionnaire propre."""

    # La marque : c'est une liste, on prend la première si elle existe
    marque = produit.brands[0].name if produit.brands else None

    # La catégorie principale
    categorie = produit.category.title if produit.category else None

    # Le prix : niché dans la première offre
    if produit.offers:
        offre = produit.offers[0]
        prix = offre.price.price
        prix_barre = offre.price.compare_at_price
        devise = offre.price.currency
        retailer = offre.domain
    else:
        prix = prix_barre = devise = retailer = None

    return {
        "id": produit.id,
        "titre": produit.title,
        "marque": marque,
        "categorie": categorie,
        "prix": prix,
        "prix_barre": prix_barre,
        "devise": devise,
        "retailer": retailer
    }

# Testons sur le premier produit
infos = extraire_infos(page.products[0])
print(infos)

In [ ]:
import pandas as pd

# Appliquer notre fonction à TOUS les produits reçus
donnees = [extraire_infos(produit) for produit in page.products]

# Transformer la liste de dictionnaires en tableau pandas
df = pd.DataFrame(donnees)

# Regarder le résultat
print("Nombre de lignes :", len(df))
df.head(10)

In [ ]:
# Les catégories qu'on veut analyser (produits mode/luxe)
requetes = [
    "leather handbag",
    "leather shoes",
    "luxury watch",
    "wool coat",
    "silk scarf",
    "sunglasses"
]

# On va collecter tous les produits ici
tous_les_produits = []

for requete in requetes:
    print(f"Recherche : {requete}...")
    page = client.products.search(query=requete)

    for produit in page.products:
        infos = extraire_infos(produit)
        infos["requete"] = requete   # on garde trace de la recherche d'origine
        tous_les_produits.append(infos)

    print(f"  → {len(page.products)} produits récupérés")

# Transformer le tout en un grand DataFrame
df = pd.DataFrame(tous_les_produits)
print("\n=== TOTAL ===")
print("Nombre total de produits :", len(df))
df.head()